# AI Resume Screening System
## Project Overview
This notebook performs data exploration and feature engineering for an AI-powered Resume Screening System.

*Objectives*
- Explore and understand the resume dataset
- Clean missing and inconsistent values
- Engineer meaningful text features
- Merge multiple data sources into a unified candidate profile
- Generate `resume_text` for downstream NLP tasks

*Dataset*
- People
- Experience
- Education
- Skills
- Abilities
- Job Descriptions

### Output
A production-ready master dataset where each row represents one candidate.

### Notebook Flow
1. Import Libraries
2. Load Dataset
3. Explore Dataset
4. Data Cleaning
5. Feature Engineering
6. Aggregate Candidate Information
7. Merge All Tables
8. Create Resume Text
9. Data Validation
10. Save Processed Dataset

### 1. Data Collection

The dataset contains multiple CSV files:
- Candidate information
- Education details
- Work experience
- Skills
- Abilities

We will combine these tables to create a complete candidate profile.

In [5]:
# Load Required Libraries
import pandas as pd
import numpy as numpy

In [6]:
# Load Raw Dataset
people=pd.read_csv("../data/raw/resumes/01_people.csv")
abilities=pd.read_csv("../data/raw/resumes/02_abilities.csv")
education=pd.read_csv("../data/raw/resumes/03_education.csv")
experience=pd.read_csv("../data/raw/resumes/04_experience.csv")
person_skills=pd.read_csv("../data/raw/resumes/05_person_skills.csv")
skills=pd.read_csv("../data/raw/resumes/06_skills.csv")

### 2. Exploratory Data Analysis (EDA)

In this step, we will do analysis of datasets and understand about:
- Dataset structure
- Missing values
- Duplicate records
- Relationship between different tables

In [7]:
#Check dataset dimensions
print("People:", people.shape)
print("Abilities:", abilities.shape)
print("Education:", education.shape)
print("Experience:", experience.shape)
print("Person Skills:", person_skills.shape)
print("Skills:", skills.shape)

People: (54933, 5)
Abilities: (1219473, 2)
Education: (75999, 5)
Experience: (265404, 6)
Person Skills: (2483376, 2)
Skills: (226760, 1)


In [8]:
# View the columns in each dataset
print("\nPeople: ",people.columns)
print("\nAbilities: ",abilities.columns)
print("\nEducation: ",education.columns)
print("\nExperience: ",experience.columns)
print("\nPerson Skills: ",person_skills.columns)
print("\nSkills: ",skills.columns)


People:  Index(['person_id', 'name', 'email', 'phone', 'linkedin'], dtype='object')

Abilities:  Index(['person_id', 'ability'], dtype='object')

Education:  Index(['person_id', 'institution', 'program', 'start_date', 'location'], dtype='object')

Experience:  Index(['person_id', 'title', 'firm', 'start_date', 'end_date', 'location'], dtype='object')

Person Skills:  Index(['person_id', 'skill'], dtype='object')

Skills:  Index(['skill'], dtype='object')


In [9]:
# Count the unique person IDs in each dataset
print("People:", people["person_id"].nunique())
print("Education:", education["person_id"].nunique())
print("Experience:", experience["person_id"].nunique())
print("Abilities:", abilities["person_id"].nunique())
print("Person Skills:", person_skills["person_id"].nunique())

People: 54933
Education: 48075
Experience: 54933
Abilities: 54930
Person Skills: 54858


In [10]:
# Inspect records for person_id = 1 across all datasets
people[people["person_id"]==1]

,person_id,name,email,phone,linkedin
0,1,Database Administrator,NaN,NaN,NaN


In [11]:
education[education["person_id"]==1]

,person_id,institution,program,start_date,location
0,1,Lead City University,Bachelor of Science,07/2013,NaN


In [12]:
experience[experience["person_id"]==1]

,person_id,title,firm,start_date,end_date,location
0,1,Database Administrator,Family Private Care LLC,04/2017,Present,"Roswell, GA"
1,1,Database Administrator,Incomm,01/2014,02/2017,"Alpharetta, GA"


In [13]:
abilities[abilities["person_id"]==1]

,person_id,ability
0,1,Installation and Building Server
1,1,Running Backups
2,1,Recovering and Restoring Models
3,1,Support various MS SQL Server
4,1,MS SQL Server 2005/2008
5,1,environments from SQL Server
6,1,/2008R2R2/2012/2014
7,1,2005 thru SQL Server 2008r2 as
8,1,administration including
9,1,well as with SQL Server 2012 on


In [14]:
person_skills[person_skills["person_id"]==1]

,person_id,skill
0,1,Database administration
1,1,Database
2,1,Ms sql server
3,1,Ms sql server 2005
4,1,Sql server
5,1,Sql server 2005
6,1,Sql server 2008
7,1,Sql server 2008 r2
8,1,Sql server 2012
9,1,Sql


### 3. Data Cleaning

Here we will slean and prepare raw data before combining different tables.

Steps performed:
- Handling missing values
- Removing unnecessary columns
- Checking duplicate records
- Standardizing text fields

In [15]:
# Check for duplicate rows in the person_skills dataset
person_skills.duplicated().sum()

np.int64(587511)

In [16]:
#Delete duplicate records from person_skills dataset
person_skills = person_skills.drop_duplicates()

In [17]:
#Verify that duplicate rows have been removed
person_skills.duplicated().sum()

np.int64(0)

In [18]:
#Check person_skills dimensions after deletion
person_skills.shape

(1895865, 2)

### 4. Data Transformation

The dataset contains multiple records for the same candidate.

Example:
A candidate can have:
- Multiple skills
- Multiple education records
- Multiple work experiences

We transform these multiple rows into a single candidate-level record.

In [19]:
# Group skills by person ID
grouped = person_skills.groupby("person_id")

In [20]:
# Check the type of the grouped object
type(grouped)

pandas.core.groupby.generic.DataFrameGroupBy

In [21]:
# Retrieve all records for person_id = 1 from the grouped data
grouped.get_group(1)

,person_id,skill
0,1,Database administration
1,1,Database
2,1,Ms sql server
3,1,Ms sql server 2005
4,1,Sql server
5,1,Sql server 2005
6,1,Sql server 2008
7,1,Sql server 2008 r2
8,1,Sql server 2012
9,1,Sql


In [22]:
# Check for missing skill values before aggregation
person_skills["skill"].isna().sum()

np.int64(6)

In [23]:
# Remove rows with missing skill values
person_skills = person_skills.dropna(subset=["skill"])

In [24]:
# Confirm there are no missing skill values
person_skills["skill"].isna().sum()

np.int64(0)

In [25]:
# Transform multiple skill records into a single row per person
skills_per_person = (
    person_skills
    .groupby("person_id")["skill"]
    .apply(lambda skills: ", ".join(pd.unique(skills)))
    .reset_index()
)

In [26]:
# Preview the aggregated skills dataset
skills_per_person.head()

,person_id,skill
0,1,"Database administration, Database, Ms sql serv..."
1,2,"sql server management studio, visual studio, s..."
2,3,"DATABASES, ORACLE (4 years), ORACLE 10G, SQL, ..."
3,4,Maintain multiple database environments (Redsh...
4,5,"Scrum, Agile software development, Product bac..."


In [27]:
# Check the shape of the aggregated skills dataset
skills_per_person.shape

(54858, 2)

In [28]:
# Display the aggregated skills for the first person
print(skills_per_person.iloc[0]["skill"])

Database administration, Database, Ms sql server, Ms sql server 2005, Sql server, Sql server 2005, Sql server 2008, Sql server 2008 r2, Sql server 2012, Sql, Sql queries, Stored procedures, Clustering, Backups, T-sql, Virtualization, R2, Maintenance, Problem solving, Shipping


In [29]:
# Aggregate unique skills for each person
skills_per_person = (
    person_skills
    .groupby("person_id")["skill"]
    .apply(lambda skills: ", ".join(pd.unique(skills)))
    .reset_index()
)

In [30]:
# Preview the updated aggregated skills dataset
skills_per_person.head()

,person_id,skill
0,1,"Database administration, Database, Ms sql serv..."
1,2,"sql server management studio, visual studio, s..."
2,3,"DATABASES, ORACLE (4 years), ORACLE 10G, SQL, ..."
3,4,Maintain multiple database environments (Redsh...
4,5,"Scrum, Agile software development, Product bac..."


In [31]:
experience[experience["person_id"] == 1]

,person_id,title,firm,start_date,end_date,location
0,1,Database Administrator,Family Private Care LLC,04/2017,Present,"Roswell, GA"
1,1,Database Administrator,Incomm,01/2014,02/2017,"Alpharetta, GA"


In [32]:
# Create a descriptive text for each work experience
experience["experience_text"] = (
    experience["title"]
    + " at "
    + experience["firm"]
    + " ("
    + experience["start_date"]
    + " - "
    + experience["end_date"]
    + ")"
)

In [33]:
#View "experience_text" column
experience["experience_text"]

0         Database Administrator at Family Private Care ...
1         Database Administrator at Incomm (01/2014 - 02...
2         Database Administrator at Intercontinental Reg...
3         Oracle Database Administrator at Cognizant (06...
4         Oracle Database Administrator at Convergys (06...
                                ...                        
265399    Python Developer at Hexaware Technologies Limi...
265400    Software Developer at Vision InfoTech Pvt Ltd ...
265401         MetroBikes at MetroBikes (09/2018 - Present)
265402    Python/Flask Developer at TechJini Solutions P...
265403    Python Developer at TechJini Solutions Pvt. Lt...
Name: experience_text, Length: 265404, dtype: object

In [34]:
experience[["person_id", "experience_text"]].head()

,person_id,experience_text
0,1,Database Administrator at Family Private Care ...
1,1,Database Administrator at Incomm (01/2014 - 02...
2,2,Database Administrator at Intercontinental Reg...
3,3,Oracle Database Administrator at Cognizant (06...
4,3,Oracle Database Administrator at Convergys (06...


In [35]:
# Check for missing values in the experience text column
experience["experience_text"].isna().sum()

np.int64(6738)

In [36]:
# Fill missing values in the experience details
experience["firm"] = experience["firm"].fillna("Unknown Company")
experience["start_date"] = experience["start_date"].fillna("Unknown Start")
experience["end_date"] = experience["end_date"].fillna("Present")
experience["title"] = experience["title"].fillna("Unknown Role")

In [37]:
# Recreate the Experience Text After Handling Missing Values
experience["experience_text"] = (
    experience["title"]
    + " at "
    + experience["firm"]
    + " ("
    + experience["start_date"]
    + " - "
    + experience["end_date"]
    + ")"
)

In [38]:
#Verify that the experience text column contains no missing values
experience["experience_text"].isna().sum()

np.int64(0)

In [39]:
# Aggregate work experience by person
experience_per_person=experience.groupby("person_id")["experience_text"].apply(lambda experiences: "; ".join(experiences)).reset_index()

In [40]:
#Check for missing values
education.isna().sum()

person_id          0
institution     1569
program         7761
start_date     21129
location       23256
dtype: int64

In [41]:
#Fill the NAN with missing values
education["institution"] = education["institution"].fillna("Unknown Institution")
education["program"] = education["program"].fillna("Unknown Program")
education["start_date"] = education["start_date"].fillna("Unknown Start")
education["location"] = education["location"].fillna("Unknown Location")

In [42]:
education.head(10)

,person_id,institution,program,start_date,location
0,1,Lead City University,Bachelor of Science,07/2013,Unknown Location
1,2,lagos state university,bsc in computer science,Unknown Start,"Lagos, GU"
2,3,"JNTU - Kakinada, Andhra Pradesh",Master of Computer Applications in Science and...,2013,"Kakinada, Andhra Pradesh"
3,4,University of Informatics,Bachelor in Computer Science,06/07,June 2007
4,5,Virginia Commomwealth University,Unknown Program,08/2013,"Richmond, VA"
5,6,School of Professional and Graduate Studies/Te...,Unknown Program,Unknown Start,"Overland Park, KS"
6,6,UNIVERSITY OF ALASKA,General/Business/Science Courses,Unknown Start,"Anchorage, AK"
7,7,University Of Yaounde,Biochemistry,04/08,Yaounde
8,8,Bowie State University,Bachelor's Degree in BiologyChem / Computer Sc...,Unknown Start,Unknown Location
9,9,Bridgewater State University,Management - Information Systems,Unknown Start,Present


In [43]:
# Create a descriptive text for each education record
education["education_info"]= (
    education["program"]    + " from "
    + education["institution"]
    + ", "
    + education["location"]
    + " ("
    + education["start_date"]
    + ")"
)

In [44]:
education["education_info"].head(10)

0    Bachelor of Science from Lead City University,...
1    bsc in computer science from lagos state unive...
2    Master of Computer Applications in Science and...
3    Bachelor in Computer Science from University o...
4    Unknown Program from Virginia Commomwealth Uni...
5    Unknown Program from School of Professional an...
6    General/Business/Science Courses from UNIVERSI...
7    Biochemistry from University Of Yaounde, Yaoun...
8    Bachelor's Degree in BiologyChem / Computer Sc...
9    Management - Information Systems from Bridgewa...
Name: education_info, dtype: object

In [45]:
#Check for duplicate education records per person
education["person_id"].duplicated().sum()

np.int64(27924)

In [46]:
#Aggregate all the education information in one group per person
education_grouped=education.groupby("person_id")["education_info"].apply(lambda ed_info:"; ".join(ed_info)).reset_index()

In [47]:
education_grouped.head()

,person_id,education_info
0,1,"Bachelor of Science from Lead City University,..."
1,2,bsc in computer science from lagos state unive...
2,3,Master of Computer Applications in Science and...
3,4,Bachelor in Computer Science from University o...
4,5,Unknown Program from Virginia Commomwealth Uni...


In [48]:
education_grouped.shape

(48075, 2)

In [49]:
grouped.get_group(1)

,person_id,skill
0,1,Database administration
1,1,Database
2,1,Ms sql server
3,1,Ms sql server 2005
4,1,Sql server
5,1,Sql server 2005
6,1,Sql server 2008
7,1,Sql server 2008 r2
8,1,Sql server 2012
9,1,Sql


In [50]:
abilities.head()

,person_id,ability
0,1,Installation and Building Server
1,1,Running Backups
2,1,Recovering and Restoring Models
3,1,Support various MS SQL Server
4,1,MS SQL Server 2005/2008


In [51]:
abilities.shape

(1219473, 2)

In [52]:
#Check for missing values
abilities.isna().sum()

person_id    0
ability      0
dtype: int64

In [53]:
#Aggregate abilities per person in one record
abilities_grouped = (
    abilities
    .groupby("person_id")["ability"]
    .apply(lambda abilities: "; ".join(abilities))
    .reset_index()
)


In [54]:
abilities_grouped.shape

(54930, 2)

In [55]:
abilities_grouped["person_id"].nunique()

54930

In [56]:
abilities_grouped.head()

,person_id,ability
0,1,Installation and Building Server; Running Back...
1,2,database management systems administration; de...
2,3,Over 4+ years of Experience as Architecture; E...
3,4,SQL management; PostgresSQL; Oracle; MySQL; mi...
4,5,Scrum Master; Agile software development; Prod...


In [57]:
#Confirm dimensions of all the datasets
skills_per_person.shape

(54858, 2)

In [58]:
experience_per_person.shape

(54933, 2)

In [59]:
education_grouped.shape

(48075, 2)

In [60]:
abilities_grouped.shape

(54930, 2)

In [61]:
#Confirm column names of all the dataset before merge
skills_per_person.columns

Index(['person_id', 'skill'], dtype='object')

In [62]:
experience_per_person.columns

Index(['person_id', 'experience_text'], dtype='object')

In [63]:
education_grouped.columns

Index(['person_id', 'education_info'], dtype='object')

In [64]:
abilities_grouped.columns

Index(['person_id', 'ability'], dtype='object')

### EDA and feature engineering Summary

experience_per_person : (54933, 2)

education_grouped     : (48075, 2)

skills_per_person     : (54858, 2)

abilities_grouped     : (54930, 2)

### 5. Data Integration

Combine all candidate information into a single master dataframe.

The final dataset contains:

- Candidate details
- Education
- Experience
- Skills
- Abilities

In [65]:
# Create the master dataset from the people table
master_df = people.copy()

master_df.head()

,person_id,name,email,phone,linkedin
0,1,Database Administrator,NaN,NaN,NaN
1,2,Database Administrator,NaN,NaN,NaN
2,3,Oracle Database Administrator,NaN,NaN,NaN
3,4,Amazon Redshift Administrator and ETL Develope...,NaN,NaN,NaN
4,5,Scrum Master Scrum Master Scrum Master,NaN,NaN,NaN


In [66]:
master_df = master_df.merge(
    experience_per_person,
    on="person_id",
    how="left"
)

In [67]:
master_df.head()

,person_id,name,email,phone,linkedin,experience_text
0,1,Database Administrator,NaN,NaN,NaN,Database Administrator at Family Private Care ...
1,2,Database Administrator,NaN,NaN,NaN,Database Administrator at Intercontinental Reg...
2,3,Oracle Database Administrator,NaN,NaN,NaN,Oracle Database Administrator at Cognizant (06...
3,4,Amazon Redshift Administrator and ETL Develope...,NaN,NaN,NaN,Amazon Redshift Administrator and ETL Develope...
4,5,Scrum Master Scrum Master Scrum Master,NaN,NaN,NaN,Scrum Master at Quest Technologies (10/2015 - ...


In [68]:
master_df = master_df.merge(
    education_grouped,
    on="person_id",
    how="left"
)
master_df.head()

,person_id,name,email,phone,linkedin,experience_text,education_info
0,1,Database Administrator,NaN,NaN,NaN,Database Administrator at Family Private Care ...,"Bachelor of Science from Lead City University,..."
1,2,Database Administrator,NaN,NaN,NaN,Database Administrator at Intercontinental Reg...,bsc in computer science from lagos state unive...
2,3,Oracle Database Administrator,NaN,NaN,NaN,Oracle Database Administrator at Cognizant (06...,Master of Computer Applications in Science and...
3,4,Amazon Redshift Administrator and ETL Develope...,NaN,NaN,NaN,Amazon Redshift Administrator and ETL Develope...,Bachelor in Computer Science from University o...
4,5,Scrum Master Scrum Master Scrum Master,NaN,NaN,NaN,Scrum Master at Quest Technologies (10/2015 - ...,Unknown Program from Virginia Commomwealth Uni...


In [69]:
master_df = master_df.merge(
    skills_per_person,
    on="person_id",
    how="left"
)
master_df.head()

,person_id,name,email,phone,linkedin,experience_text,education_info,skill
0,1,Database Administrator,NaN,NaN,NaN,Database Administrator at Family Private Care ...,"Bachelor of Science from Lead City University,...","Database administration, Database, Ms sql serv..."
1,2,Database Administrator,NaN,NaN,NaN,Database Administrator at Intercontinental Reg...,bsc in computer science from lagos state unive...,"sql server management studio, visual studio, s..."
2,3,Oracle Database Administrator,NaN,NaN,NaN,Oracle Database Administrator at Cognizant (06...,Master of Computer Applications in Science and...,"DATABASES, ORACLE (4 years), ORACLE 10G, SQL, ..."
3,4,Amazon Redshift Administrator and ETL Develope...,NaN,NaN,NaN,Amazon Redshift Administrator and ETL Develope...,Bachelor in Computer Science from University o...,Maintain multiple database environments (Redsh...
4,5,Scrum Master Scrum Master Scrum Master,NaN,NaN,NaN,Scrum Master at Quest Technologies (10/2015 - ...,Unknown Program from Virginia Commomwealth Uni...,"Scrum, Agile software development, Product bac..."


In [70]:
master_df = master_df.merge(
    abilities_grouped,
    on="person_id",
    how="left"
)
master_df.head()

,person_id,name,email,phone,linkedin,experience_text,education_info,skill,ability
0,1,Database Administrator,NaN,NaN,NaN,Database Administrator at Family Private Care ...,"Bachelor of Science from Lead City University,...","Database administration, Database, Ms sql serv...",Installation and Building Server; Running Back...
1,2,Database Administrator,NaN,NaN,NaN,Database Administrator at Intercontinental Reg...,bsc in computer science from lagos state unive...,"sql server management studio, visual studio, s...",database management systems administration; de...
2,3,Oracle Database Administrator,NaN,NaN,NaN,Oracle Database Administrator at Cognizant (06...,Master of Computer Applications in Science and...,"DATABASES, ORACLE (4 years), ORACLE 10G, SQL, ...",Over 4+ years of Experience as Architecture; E...
3,4,Amazon Redshift Administrator and ETL Develope...,NaN,NaN,NaN,Amazon Redshift Administrator and ETL Develope...,Bachelor in Computer Science from University o...,Maintain multiple database environments (Redsh...,SQL management; PostgresSQL; Oracle; MySQL; mi...
4,5,Scrum Master Scrum Master Scrum Master,NaN,NaN,NaN,Scrum Master at Quest Technologies (10/2015 - ...,Unknown Program from Virginia Commomwealth Uni...,"Scrum, Agile software development, Product bac...",Scrum Master; Agile software development; Prod...


In [71]:
#Final shape of master_df after merge
master_df.shape

(54933, 9)

In [72]:
master_df.head()

,person_id,name,email,phone,linkedin,experience_text,education_info,skill,ability
0,1,Database Administrator,NaN,NaN,NaN,Database Administrator at Family Private Care ...,"Bachelor of Science from Lead City University,...","Database administration, Database, Ms sql serv...",Installation and Building Server; Running Back...
1,2,Database Administrator,NaN,NaN,NaN,Database Administrator at Intercontinental Reg...,bsc in computer science from lagos state unive...,"sql server management studio, visual studio, s...",database management systems administration; de...
2,3,Oracle Database Administrator,NaN,NaN,NaN,Oracle Database Administrator at Cognizant (06...,Master of Computer Applications in Science and...,"DATABASES, ORACLE (4 years), ORACLE 10G, SQL, ...",Over 4+ years of Experience as Architecture; E...
3,4,Amazon Redshift Administrator and ETL Develope...,NaN,NaN,NaN,Amazon Redshift Administrator and ETL Develope...,Bachelor in Computer Science from University o...,Maintain multiple database environments (Redsh...,SQL management; PostgresSQL; Oracle; MySQL; mi...
4,5,Scrum Master Scrum Master Scrum Master,NaN,NaN,NaN,Scrum Master at Quest Technologies (10/2015 - ...,Unknown Program from Virginia Commomwealth Uni...,"Scrum, Agile software development, Product bac...",Scrum Master; Agile software development; Prod...


In [73]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54933 entries, 0 to 54932
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   person_id        54933 non-null  int64 
 1   name             54819 non-null  object
 2   email            1593 non-null   object
 3   phone            1833 non-null   object
 4   linkedin         8538 non-null   object
 5   experience_text  54933 non-null  object
 6   education_info   48075 non-null  object
 7   skill            54858 non-null  object
 8   ability          54930 non-null  object
dtypes: int64(1), object(8)
memory usage: 3.8+ MB


In [74]:
# Check for missing values in the merged columns
master_df[
    [
        "experience_text",
        "education_info",
        "skill",
        "ability"
    ]
].isnull().sum()

experience_text       0
education_info     6858
skill                75
ability               3
dtype: int64

In [75]:
# Replace missing values with empty strings
master_df = master_df.fillna("")

### 6. Feature Engineering

Objective: Convert structured candidate information into a single textual feature that can be used for NLP-based similarity matching.

Features combined:

- Experience
- Education
- Skills
- Abilities

The final feature:
`resume_text`

In [76]:
# Create the consolidated resume text
master_df["resume_text"] = (
    master_df["name"] + " " +
    master_df["experience_text"] + " " +
    master_df["education_info"] + " " +
    master_df["skill"] + " " +
    master_df["ability"]
)

In [77]:
master_df["resume_text"].head()

0    Database Administrator Database Administrator ...
1    Database Administrator Database Administrator ...
2    Oracle Database Administrator Oracle Database ...
3    Amazon Redshift Administrator and ETL Develope...
4    Scrum Master Scrum Master Scrum Master Scrum M...
Name: resume_text, dtype: object

In [80]:
master_df[["name", "resume_text"]].head(3)

,name,resume_text
0,Database Administrator,Database Administrator Database Administrator ...
1,Database Administrator,Database Administrator Database Administrator ...
2,Oracle Database Administrator,Oracle Database Administrator Oracle Database ...


In [81]:
master_df.head()

,person_id,name,email,phone,linkedin,experience_text,education_info,skill,ability,resume_text
0,1,Database Administrator,,,,Database Administrator at Family Private Care ...,"Bachelor of Science from Lead City University,...","Database administration, Database, Ms sql serv...",Installation and Building Server; Running Back...,Database Administrator Database Administrator ...
1,2,Database Administrator,,,,Database Administrator at Intercontinental Reg...,bsc in computer science from lagos state unive...,"sql server management studio, visual studio, s...",database management systems administration; de...,Database Administrator Database Administrator ...
2,3,Oracle Database Administrator,,,,Oracle Database Administrator at Cognizant (06...,Master of Computer Applications in Science and...,"DATABASES, ORACLE (4 years), ORACLE 10G, SQL, ...",Over 4+ years of Experience as Architecture; E...,Oracle Database Administrator Oracle Database ...
3,4,Amazon Redshift Administrator and ETL Develope...,,,,Amazon Redshift Administrator and ETL Develope...,Bachelor in Computer Science from University o...,Maintain multiple database environments (Redsh...,SQL management; PostgresSQL; Oracle; MySQL; mi...,Amazon Redshift Administrator and ETL Develope...
4,5,Scrum Master Scrum Master Scrum Master,,,,Scrum Master at Quest Technologies (10/2015 - ...,Unknown Program from Virginia Commomwealth Uni...,"Scrum, Agile software development, Product bac...",Scrum Master; Agile software development; Prod...,Scrum Master Scrum Master Scrum Master Scrum M...


### NLP Text Preprocessing

In this step, we clean and normalize resume text before converting it into numerical representations.

Steps:
- Convert text to lowercase
- Remove unwanted characters
- Remove extra spaces
- Prepare text for feature extraction